In [218]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ucphm import *
from functions.ct_column import *
from functions.ensembles import *
from scipy.stats import skew, shapiro

In [219]:
dataset = pd.read_excel('../../../BreastCareCond.xlsx')
data_code = pd.read_excel('../../../Codificação variaveis.xlsx')

Using the variables she used

This also as the surgery data, the only data that appeared here were this three variables 'Type of 1st intervention', 'Axillary operation', 'SLN procedure'

I changed the n regional nodes affected to be a numerical variable

In [220]:
#Defining the proposed has good treatment variables to use (the empty ones were removed)
relevant_variables = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC']
numerical_variables = ['Age', 'Ki67 positive cells (%) at CB', 'N (Regional nodes affected)']
categorical_variables = ['Number identifying the lesion in the patient', 
                         'Side location of the lesion', 'Classification with respect to other lesions', 
                         'Source of referral', 'Type of invasion at CB', 'Istotype at CB', 'Grade at CB', 
                         'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 
                         'Her2 overexpression (with immunohystochemistry) at CB', 'Disease extent', 
                         
                         'Type of 1st intervention', 'Axillary operation', 'SLN procedure',  # these latter ones (last three) are surgery related ones
                         
                         'Final pathology', 'Final recommendations to patient'] # these are for the ct neoadjuvant analysis
                         
date_columns = ['Date of CB ', 'Date of 1st breast operation', 'Date of intervention on sentinel lymph node',
                'Date of intervention (axilla)']
selected_variables = relevant_variables + date_columns + numerical_variables + categorical_variables

In [221]:
# Variables to check
variables_to_check = ['Type of invasion at CB', 'Grade at CB', 
                      'Oestrogen receptor status at CB', 
                      'Progesterone receptor status at CB', 
                      'Her2 overexpression (with immunohystochemistry) at CB', 'Disease extent']

# Subset the dataset with the specified variables and PERSON_ID
subset = dataset[['PERSON_ID'] + variables_to_check]

# For each patient with at least one NaN, count how many of the variables are missing
nan_counts = subset[subset[variables_to_check].isna().any(axis=1)].copy()
nan_counts['num_nan'] = nan_counts[variables_to_check].isna().sum(axis=1)

# Count number of patients by number of NaNs (from 5 to 1)
nan_summary = nan_counts['num_nan'].value_counts().sort_index(ascending=False)
for n in range(5, 0, -1):
    count = nan_summary.get(n, 0)
    print(f"Number of patients with {n} NaN values: {count}")

Number of patients with 5 NaN values: 297
Number of patients with 4 NaN values: 4
Number of patients with 3 NaN values: 23
Number of patients with 2 NaN values: 376
Number of patients with 1 NaN values: 329


Replace missing values of diagnostic and surgery categorical first

In [222]:
'''replace_zero = ['N (Regional nodes affected)', 'Type of invasion at CB', 'Grade at CB', 
                'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 'Her2 overexpression (with immunohystochemistry) at CB']
replace_one = ['Classification with respect to other lesions']
replace_four = ['Istotype at CB', 'Disease extent']
replace_nine = ['Type of 1st intervention']
replace_missing_values(dataset, replace_zero, 0)
replace_missing_values(dataset, replace_one, 1)
replace_missing_values(dataset, replace_four, 4)
replace_missing_values(dataset, replace_nine, 9)
# replace the missing values with the mode of the column
mode_value = dataset['Side location of the lesion'].mode()[0]
dataset['Side location of the lesion'] = dataset['Side location of the lesion'].fillna(mode_value)'''

"replace_zero = ['N (Regional nodes affected)', 'Type of invasion at CB', 'Grade at CB', \n                'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 'Her2 overexpression (with immunohystochemistry) at CB']\nreplace_one = ['Classification with respect to other lesions']\nreplace_four = ['Istotype at CB', 'Disease extent']\nreplace_nine = ['Type of 1st intervention']\nreplace_missing_values(dataset, replace_zero, 0)\nreplace_missing_values(dataset, replace_one, 1)\nreplace_missing_values(dataset, replace_four, 4)\nreplace_missing_values(dataset, replace_nine, 9)\n# replace the missing values with the mode of the column\nmode_value = dataset['Side location of the lesion'].mode()[0]\ndataset['Side location of the lesion'] = dataset['Side location of the lesion'].fillna(mode_value)"

In [223]:
replace_zero = ['N (Regional nodes affected)']
replace_one = ['Classification with respect to other lesions']
replace_four = ['Istotype at CB', 'Disease extent']
replace_nine = ['Type of 1st intervention']
replace_missing_values(dataset, replace_zero, 0)
replace_missing_values(dataset, replace_one, 1)
replace_missing_values(dataset, replace_four, 4)
replace_missing_values(dataset, replace_nine, 9)
# replace the missing values with the mode of the column
mode_value = dataset['Side location of the lesion'].mode()[0]
dataset['Side location of the lesion'] = dataset['Side location of the lesion'].fillna(mode_value)

Done
Done
Done
Done


In [224]:
dataset.dropna(subset=['Type of invasion at CB', 'Grade at CB', 
                      'Oestrogen receptor status at CB', 
                      'Progesterone receptor status at CB', 
                      'Her2 overexpression (with immunohystochemistry) at CB'], inplace=True)

dataset.reset_index(drop=True, inplace=True)

Creating the cT variable like she did

In [225]:
grouped_T(dataset)
cT (dataset)
clinical_staging (dataset)
new_variables = ['cT']
selected_variables = selected_variables + new_variables
categorical_variables = categorical_variables + new_variables
dataset['cT'] = dataset['cT'].replace('TX', np.nan)

In [226]:
change_to_numeric(dataset, numerical_variables)
change_to_categorical(dataset, categorical_variables)

Done
Done


In [227]:
cT = ['cT']
replace_missing_values(dataset, cT, 'unknown')

Done


To reduce the number of values

In [228]:
'''simplify_2 = ['N (Regional nodes affected)', 'Number identifying the lesion in the patient']
simplify_3 = ['Istotype at CB']
simplify_variable(dataset, simplify_2, 2)
simplify_variable(dataset, simplify_3, 3)'''

"simplify_2 = ['N (Regional nodes affected)', 'Number identifying the lesion in the patient']\nsimplify_3 = ['Istotype at CB']\nsimplify_variable(dataset, simplify_2, 2)\nsimplify_variable(dataset, simplify_3, 3)"

In [229]:
test_normality(dataset, numerical_variables)

,Variable,Skewness,Shapiro p-value,Normality,Has Outliers
0,Age,0.217506,1.424405e-15,Probably not normal,False
1,Ki67 positive cells (%) at CB,0.978065,9.293414e-41,Probably not normal,True
2,N (Regional nodes affected),2.096774,2.442320e-66,Probably not normal,True


In [230]:
median_imputer(dataset, numerical_variables)

'Done'

To do label encoding, I hust did it for a variable that did not removed information

In [231]:
'''dataset['Number identifying the lesion in the patient'] = dataset['Number identifying the lesion in the patient'].map({'1' : 0, '>1' : 1})
dataset['Side location of the lesion'] = dataset['Side location of the lesion'].map({'L' : 0,'R' : 1})
dataset['Source of referral'] = dataset['Source of referral'].map({1 : 0, 2 : 1})
dataset['N (Regional nodes affected)'] = dataset['N (Regional nodes affected)'].map({'0.0' : 0, '1.0' : 1, '>1' : 2})
dataset['Istotype at CB'] = dataset['Istotype at CB'].map({'1.0' : 1, '2.0' : 2, '>2' : 3})'''

"dataset['Number identifying the lesion in the patient'] = dataset['Number identifying the lesion in the patient'].map({'1' : 0, '>1' : 1})\ndataset['Side location of the lesion'] = dataset['Side location of the lesion'].map({'L' : 0,'R' : 1})\ndataset['Source of referral'] = dataset['Source of referral'].map({1 : 0, 2 : 1})\ndataset['N (Regional nodes affected)'] = dataset['N (Regional nodes affected)'].map({'0.0' : 0, '1.0' : 1, '>1' : 2})\ndataset['Istotype at CB'] = dataset['Istotype at CB'].map({'1.0' : 1, '2.0' : 2, '>2' : 3})"

In [232]:
dataset['Side location of the lesion'] = dataset['Side location of the lesion'].map({'L' : 0,'R' : 1})
dataset['Source of referral'] = dataset['Source of referral'].map({1 : 0, 2 : 1})

In [233]:
'''one_hot_variables = ['N (Regional nodes affected)', 'Classification with respect to other lesions', 'Type of invasion at CB', 'Istotype at CB', 'Grade at CB', 
                     'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 'Her2 overexpression (with immunohystochemistry) at CB', 
                     'Disease extent', 'cT', 'Type of 1st intervention']
ohe_dataset, new_column_names = one_hot_encoding(dataset, one_hot_variables)'''

"one_hot_variables = ['N (Regional nodes affected)', 'Classification with respect to other lesions', 'Type of invasion at CB', 'Istotype at CB', 'Grade at CB', \n                     'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 'Her2 overexpression (with immunohystochemistry) at CB', \n                     'Disease extent', 'cT', 'Type of 1st intervention']\nohe_dataset, new_column_names = one_hot_encoding(dataset, one_hot_variables)"

here I removed the nodes and added the number identifiyng the leasion

In [234]:
one_hot_variables = ['Number identifying the lesion in the patient', 'Classification with respect to other lesions', 'Istotype at CB',
                     'Disease extent', 'cT', 'Type of 1st intervention']
one_hot_variables_drop = []
ohe_dataset, new_column_names = one_hot_encoding(dataset, one_hot_variables)

In [235]:
ohe_dataset = date_of_diagnosis(ohe_dataset, date_columns)

In [236]:
selected_variables = [col for col in selected_variables if col not in one_hot_variables]
selected_variables = [col for col in selected_variables if col not in date_columns]
selected_variables.extend(new_column_names.tolist())
date_of_diagnosis = ['Date of diagnosis']
selected_variables = selected_variables + date_of_diagnosis

categorical_variables = [col for col in categorical_variables if col not in one_hot_variables]
categorical_variables.extend(new_column_names.tolist())

treatment_variables = ['Radiotherapy (RT) performed', 'Conventional RT fraction', 'Total administered dose', 'Boost dose administered', 
                       'Radiotherapy on chest wall', 'Radiotherapy on supraclavicular area', 'Radiotherapy on internal mammary chain',
                        'Endocrine therapy performed', 'State (if hormone therapy prescribed)', 'Treatment in association with chemotherapy', 
                        'Chemotherapy performed', 'Adjuvant chemotherapy', 'Adjuvant therapy with biological drugs']
selected_variables_treat = selected_variables + treatment_variables

survival_dataset = ohe_dataset[selected_variables_treat].copy()

In [237]:
relevant_dates = ['Date of diagnosis', 'Date of 1st regional recurrence']
survival_dataset = remove_zeros_na(survival_dataset, relevant_dates)
survival_dataset = format_date(survival_dataset, relevant_dates)

In [238]:
survival_dataset = survival_days(survival_dataset, ['Date of diagnosis'], ['Date of 1st regional recurrence'])
survival_dataset = tte_negative_patients(survival_dataset)
survival_dataset = filter_EBC(survival_dataset)
survival_dataset.reset_index(drop=True, inplace=True)

Now the treatment variables, just because it was already made

In [239]:
dataset = survival_dataset

In [240]:
numerical_data, categorical_data = type_separation(data_code, treatment_variables)
len(numerical_data), len(categorical_data)

(3, 10)

In [241]:
replace_missing_values(dataset, numerical_data, 0)

Done


In [242]:
'''# Just because this one is different
def simplify_variable_6(dataset, variables, max_value):
    for variable in variables:
        dataset[variable] = dataset[variable].apply(
            lambda x: str(x) if x < max_value else '>6'
        )
    return dataset'''

"# Just because this one is different\ndef simplify_variable_6(dataset, variables, max_value):\n    for variable in variables:\n        dataset[variable] = dataset[variable].apply(\n            lambda x: str(x) if x < max_value else '>6'\n        )\n    return dataset"

In [243]:
'''one_hot_variables_2 = ['Conventional RT fraction', 'Boost dose administered']
dataset = simplify_variable_6(dataset, one_hot_variables_2, 4)
change_to_categorical(dataset, one_hot_variables_2)'''

"one_hot_variables_2 = ['Conventional RT fraction', 'Boost dose administered']\ndataset = simplify_variable_6(dataset, one_hot_variables_2, 4)\nchange_to_categorical(dataset, one_hot_variables_2)"

In [244]:
dataset.drop(index=0, inplace=True)
dataset.reset_index(inplace=True, drop=True)
replace_missing_values(dataset, categorical_data, 0)

Done


In [245]:
'''one_hot_variables = ['Endocrine therapy performed','Adjuvant chemotherapy', 'Chemotherapy performed']
def two_to_zero(dataset, variables):
    for variable in variables:
        dataset[variable] = dataset[variable].replace(2, 0)
    return 'Done'

two_to_zero(dataset, one_hot_variables)

bio_variable = 'Adjuvant therapy with biological drugs'
df = encode(dataset, one_hot_variables, 'first')
df_2 = encode(df, one_hot_variables_2, None)
one_hot_dataset = encode_bio_therapy(df_2, bio_variable)'''

"one_hot_variables = ['Endocrine therapy performed','Adjuvant chemotherapy', 'Chemotherapy performed']\ndef two_to_zero(dataset, variables):\n    for variable in variables:\n        dataset[variable] = dataset[variable].replace(2, 0)\n    return 'Done'\n\ntwo_to_zero(dataset, one_hot_variables)\n\nbio_variable = 'Adjuvant therapy with biological drugs'\ndf = encode(dataset, one_hot_variables, 'first')\ndf_2 = encode(df, one_hot_variables_2, None)\none_hot_dataset = encode_bio_therapy(df_2, bio_variable)"

In [246]:
one_hot_variables = ['Endocrine therapy performed','Adjuvant chemotherapy', 'Chemotherapy performed']
def two_to_zero(dataset, variables):
    for variable in variables:
        dataset[variable] = dataset[variable].replace(2, 0)
    return 'Done'

two_to_zero(dataset, one_hot_variables)

bio_variable = 'Adjuvant therapy with biological drugs'
df = encode(dataset, one_hot_variables, 'first')
one_hot_dataset = encode_bio_therapy(df, bio_variable)

In [247]:
'''feature_columns = ['Endocrine therapy performed_1', 'Adjuvant chemotherapy_1',
                  'Chemotherapy performed_1', 'Adjuvant therapy with bio drugs', 'Trastuzumab or other drugs', 
                  'Conventional RT fraction_2.0', 'Conventional RT fraction_3.0', 'Conventional RT fraction_>6', 'Conventional RT fraction_0.0', 
                  'Boost dose administered_2.0', 'Boost dose administered_3.0', 'Boost dose administered_>6',	'Boost dose administered_0.0']
feature_columns = feature_columns + categorical_data + numerical_data
bio_variable = ['Adjuvant therapy with biological drugs']
one_hot_variables = one_hot_variables + one_hot_variables_2 + bio_variable
treatment_variables = [col for col in feature_columns if col not in one_hot_variables]'''

"feature_columns = ['Endocrine therapy performed_1', 'Adjuvant chemotherapy_1',\n                  'Chemotherapy performed_1', 'Adjuvant therapy with bio drugs', 'Trastuzumab or other drugs', \n                  'Conventional RT fraction_2.0', 'Conventional RT fraction_3.0', 'Conventional RT fraction_>6', 'Conventional RT fraction_0.0', \n                  'Boost dose administered_2.0', 'Boost dose administered_3.0', 'Boost dose administered_>6',\t'Boost dose administered_0.0']\nfeature_columns = feature_columns + categorical_data + numerical_data\nbio_variable = ['Adjuvant therapy with biological drugs']\none_hot_variables = one_hot_variables + one_hot_variables_2 + bio_variable\ntreatment_variables = [col for col in feature_columns if col not in one_hot_variables]"

In [248]:
feature_columns = ['Endocrine therapy performed_1', 'Adjuvant chemotherapy_1',
                  'Chemotherapy performed_1', 'Adjuvant therapy with bio drugs', 'Trastuzumab or other drugs']
feature_columns = feature_columns + categorical_data + numerical_data
bio_variable = ['Adjuvant therapy with biological drugs']
one_hot_variables = one_hot_variables + bio_variable
treatment_variables = [col for col in feature_columns if col not in one_hot_variables]

In [249]:
to_remove = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC', 'Date of diagnosis']
important_variables = ['PERSON_ID', 'Regional LN recurrence', 'Survival days']
diagnosis_variables = [col for col in selected_variables if col not in to_remove]
selected_columns = important_variables + treatment_variables + diagnosis_variables

In [250]:
survival_dataset = one_hot_dataset[selected_columns]

In [251]:
surgery_data = ['Axillary operation', 'SLN procedure', 'Type of 1st intervention_2.0', 'Type of 1st intervention_3.0', 
                'Type of 1st intervention_6.0', 'Type of 1st intervention_8.0', 'Type of 1st intervention_9.0']

In [252]:
# remove surgery-related columns from treatment and diagnostic variable lists
surgery_cols = surgery_data  # uses existing variable from earlier cells

treatment_variables = [c for c in treatment_variables if c not in surgery_cols]
diagnosis_variables = [c for c in diagnosis_variables if c not in surgery_cols]

# quick confirmation
print("Removed surgery columns:", surgery_cols)
print("Remaining treatment_variables:", len(treatment_variables))
print("Remaining diagnosis_variables:", len(diagnosis_variables))

Removed surgery columns: ['Axillary operation', 'SLN procedure', 'Type of 1st intervention_2.0', 'Type of 1st intervention_3.0', 'Type of 1st intervention_6.0', 'Type of 1st intervention_8.0', 'Type of 1st intervention_9.0']
Remaining treatment_variables: 14
Remaining diagnosis_variables: 34


In [254]:
with open("diagnosis_nan.txt", "w") as file:
    for item in diagnosis_variables:
        file.write(str(item) + "\n")
with open("treatment_nan.txt", "w") as file:
    for item in treatment_variables:
        file.write(str(item) + "\n")
with open("surgery_nan.txt", "w") as file:
    for item in surgery_data:
        file.write(str(item) + "\n")

In [255]:
survival_dataset.to_csv("data_treated_nan.csv")